# ML-07 — Baseline Action Score and Top-10 Review

This baseline prioritizes content pages for review using only signals available in the starter snapshot.

**Decision:** which pages should a content/SEO team review first?

**Important:** `trend_direction`, `trend_pct`, and `is_declining_label` are evaluation-only. They are never used to construct the score.


## 0. Load the starter data

This notebook is designed to run directly in Colab from the GitHub notebook. If the repository is not the current working directory, it automatically clones the public repository and uses the starter CSV from there.

The dataset is not uploaded or written back to GitHub.


In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/engyusufayman06/ml-internship-2026.git"
REPO_DIR = Path("/content/ml-internship-2026")

candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("/content/ml-internship-2026/data/raw/content_refresh_anonymized.csv"),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)

if DATA_PATH is None and not REPO_DIR.exists():
    print("Repository not found in the current Colab runtime. Cloning...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

DATA_PATH = next((p for p in [Path("data/raw/content_refresh_anonymized.csv"), REPO_DIR / "data/raw/content_refresh_anonymized.csv"] if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("Starter dataset not found. Expected data/raw/content_refresh_anonymized.csv in the repository.")

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH.resolve())
print("Shape:", df.shape)

required = {"content_id", "days_since_last_update", "impressions_90d", "avg_position", "is_declining_label", "trend_direction", "trend_pct"}
missing = required - set(df.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"
target = "is_declining_label"
print("Target available for retrospective evaluation only:", target)


## 1. Signal checks and rule reasoning

### Signal 1 — Freshness / staleness
**Hypothesis:** older pages since their last update are more likely to need review.

**FlyRank flag link:** this is the signal behind refresh/staleness flags.

### Signal 2 — Search visibility / volume
**Hypothesis:** pages with more impressions are higher-value review opportunities than pages with little search visibility.

**FlyRank flag link:** this is the volume signal behind quick-win logic.


In [ ]:
fresh_bins = [-np.inf, 30, 90, 180, 365, np.inf]
fresh_labels = ["0-30", "31-90", "91-180", "181-365", "365+"]
df["freshness_bucket"] = pd.cut(df["days_since_last_update"], bins=fresh_bins, labels=fresh_labels, right=True, include_lowest=True)
fresh_check = df.groupby("freshness_bucket", observed=False)[target].agg(n="size", declining_rate="mean").reset_index()
print("Signal 1 — freshness bucket table (n printed):")
print(fresh_check.to_string(index=False))

vol_bins = [-np.inf, 0, 100, 1000, 3000, 30000, np.inf]
vol_labels = ["0", "1-100", "101-1k", "1k-3k", "3k-30k", "30k+"]
df["volume_bucket"] = pd.cut(df["impressions_90d"], bins=vol_bins, labels=vol_labels, right=True, include_lowest=True)
volume_check = df.groupby("volume_bucket", observed=False)[target].agg(n="size", declining_rate="mean").reset_index()
print("\nSignal 2 — impression bucket table (n printed):")
print(volume_check.to_string(index=False))

fresh_rate = fresh_check.set_index("freshness_bucket")["declining_rate"]
fresh_delta = fresh_rate.get("181-365", np.nan) - fresh_rate.get("0-30", np.nan)
vol_rate = volume_check.set_index("volume_bucket")["declining_rate"]
vol_delta = vol_rate.get("3k-30k", np.nan) - vol_rate.get("1-100", np.nan)

def verdict(delta):
    if pd.isna(delta): return "FALSE"
    if delta > 0: return "CONFIRMED"
    if delta < 0: return "OPPOSITE"
    return "MIXED"

fresh_verdict = verdict(fresh_delta)
volume_verdict = verdict(vol_delta)
print(f"\nSignal 1 verdict: {fresh_verdict}")
print(f"Reason: 181-365 declining rate minus 0-30 rate = {fresh_delta:+.3f}.")
print(f"Signal 2 verdict: {volume_verdict}")
print(f"Reason: 3k-30k declining rate minus 1-100 rate = {vol_delta:+.3f}.")


### Rule in plain words

**Review a page first when it is both stale (at least 180 days since its last update) and visibly receiving search impressions (at least 3,000 impressions in 90 days).**

The score is the page's 90-day impressions when both conditions are true; otherwise the score is zero.


## 2. Build the ranked queue

The queue contains one row per content item. Columns: `content_id`, `score`, `reason_code`, `action_label`.


In [ ]:
stale = df["days_since_last_update"].fillna(0) >= 180
visible = df["impressions_90d"].fillna(0) >= 3000
selected = stale & visible

df["score"] = np.where(selected, df["impressions_90d"].fillna(0), 0.0)
df["reason_code"] = np.where(selected, "stale_but_visible", "not_flagged")
df["action_label"] = np.where(selected, "refresh_review", "monitor")

queue = df[["content_id", "score", "reason_code", "action_label"]].sort_values(["score", "content_id"], ascending=[False, True]).reset_index(drop=True)

repo_root = REPO_DIR if REPO_DIR.exists() else Path.cwd()
out_dir = repo_root / "work/outputs"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "baseline_action_score.csv"
queue.to_csv(out_path, index=False)

print("Rows ranked:", len(queue))
print("Flagged for refresh review:", int(selected.sum()))
print("Queue written to:", out_path.resolve())
print("\nTop 10:")
print(queue.head(10).to_string(index=False))


## Baseline evaluation

Precision@K is a retrospective check against the observed label. The label is **not** an input to the rule.


In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df[target].mean()
print(f"Base rate (observed declining label): {base_rate:.3f}")
for k in [20, 50]:
    print(f"Precision@{k}: {precision_at_k(df['score'], df[target], k):.3f}")


## 3. Top-10 review

For each of the top ten rows: state the action, why it is there, and what would make the pick wrong.


In [ ]:
top10 = queue.head(10).merge(df[["content_id", "days_since_last_update", "impressions_90d", "avg_position"]], on="content_id", how="left")
for i, row in top10.iterrows():
    wrong = ("Wrong if the page is already accurate/current, the traffic is not commercially useful, or the apparent visibility is a measurement artifact.")
    print(f"{i+1}. {row['content_id']} — action: {row['action_label']}; why: {row['reason_code']} (age={row['days_since_last_update']:.0f}d, impressions_90d={row['impressions_90d']:.0f}, score={row['score']:.0f}); what would make it wrong: {wrong}")


## 4. Weak picks + leakage self-check


In [ ]:
weak = queue[queue["score"] > 0].tail(5)
print("Weak non-zero picks:")
print(weak.to_string(index=False))

score_inputs = {"days_since_last_update", "impressions_90d"}
forbidden = {"trend_direction", "trend_pct", "is_declining_label"}
assert score_inputs.isdisjoint(forbidden)
assert target in df.columns
assert out_path.exists()
print("\nLeakage check: PASS")
print("Rule inputs:", sorted(score_inputs))
print("Forbidden label-source fields were not used in score construction.")
print("Required CSV exists:", out_path.exists())


## 5. Self-check

- [x] Two signal checks with visible bucket tables and `n`.
- [x] At least one signal is directly linked to a FlyRank flag.
- [x] One transparent rule with a score, reason code, and action label.
- [x] Ranked queue written to `work/outputs/baseline_action_score.csv`.
- [x] Top-10 review includes action, why, and what would make each pick wrong.
- [x] No future-window or label-derived inputs in the rule.
- [x] Label is used only for retrospective evaluation.
- [x] Dataset is not uploaded or written back to GitHub by this notebook.
